In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ========================================
# 把 ZecZec_Group_Data 底下所有的 projects_summary.csv
# 合併寫進同一個 zeczec_history.db（SQLite），存放在 ZecZec_Group_Data 底下
#
# 用法：直接複製貼到你 Colab 裡新的一個 cell 執行即可
# （如果你的 notebook 前面已經定義過 BASE_DIR，這裡會直接沿用；
#  如果是獨立執行，會用下面的預設路徑，記得改成你實際的路徑）
# ========================================
import os
import sqlite3
import pandas as pd

# 沿用既有的 BASE_DIR（如果你在同一個 Colab session 裡已經跑過主流程 notebook）
# 沒有的話就用這個預設路徑，記得改成你實際的 Google Drive 路徑
BASE_DIR = '/content/drive/MyDrive/ZecZec_Group_Data'


DB_PATH = os.path.join(BASE_DIR, 'zeczec_history.db')
TABLE_NAME = 'projects_summary'
TARGET_FILENAME = 'projects_summary.csv'

# 1) 只掃「第一層」資料夾（例如 出版_New_ZecZec_Dataset、社會_New_ZecZec_Dataset...），
#    不用 os.walk 整棵樹遞迴掃到底下每個專案、每張圖片，快非常多
category_dirs = [
    d for d in os.listdir(BASE_DIR)
    if os.path.isdir(os.path.join(BASE_DIR, d))
]

csv_files = []
for d in category_dirs:
    candidate = os.path.join(BASE_DIR, d, TARGET_FILENAME)
    if os.path.isfile(candidate):
        csv_files.append(candidate)

print(f"📂 掃描完成（僅第一層），共找到 {len(csv_files)} 個 {TARGET_FILENAME}")

if not csv_files:
    raise RuntimeError(f"❌ 在 {BASE_DIR} 底下找不到任何 {TARGET_FILENAME}，請確認路徑或檔名。")

# 2) 逐一讀取，補上來源資訊欄位，方便之後追溯是哪個資料夾/檔案來的
dfs = []
failed = []
for fpath in csv_files:
    try:
        df = pd.read_csv(fpath)
    except Exception as e:
        print(f"  ⚠️ 讀取失敗，略過：{fpath}（{e}）")
        failed.append(fpath)
        continue

    df['_source_folder'] = os.path.relpath(os.path.dirname(fpath), BASE_DIR)
    df['_source_file'] = os.path.relpath(fpath, BASE_DIR)
    dfs.append(df)

print(f"✅ 成功讀取 {len(dfs)} 個檔案（失敗 {len(failed)} 個）")

# 3) 合併所有 CSV
#    不同資料夾的 CSV 欄位不完全一樣也沒關係，pandas 會自動用聯集對齊欄位，
#    缺欄位的地方會補 NaN。
combined = pd.concat(dfs, ignore_index=True, sort=False)
print(f"📊 合併完成，共 {len(combined)} 筆資料，欄位共 {len(combined.columns)} 個：")
print(f"   {list(combined.columns)}")

# 4) 寫進 SQLite 資料庫（放在 ZecZec_Group_Data 底下）
#    if_exists='replace'：每次執行都會重建這個資料表（用最新掃描結果整批覆蓋）。
#    如果你想改成「累加」而不是每次重建，把這裡改成 if_exists='append'，
#    但要注意會重複累加同樣的資料，除非自己另外處理去重。
conn = sqlite3.connect(DB_PATH)
combined.to_sql(TABLE_NAME, conn, if_exists='replace', index=False)
conn.close()

print(f"\n🎉 已寫入完成：{DB_PATH}")
print(f"   資料表名稱：{TABLE_NAME}（共 {len(combined)} 筆）")

if failed:
    print(f"\n⚠️ 以下 {len(failed)} 個檔案讀取失敗，未納入資料庫：")
    for f in failed:
        print(f"   - {f}")

📂 掃描完成（僅第一層），共找到 8 個 projects_summary.csv
✅ 成功讀取 8 個檔案（失敗 0 個）
📊 合併完成，共 2104 筆資料，欄位共 14 個：
   ['主分類', '次分類', '募資狀態', '專案編號', '方案價格列表', '折扣層數', 'FAQ總題數', 'FAQ更新頻率', '開始日期', '結束日期', '總天數', '達標率(%)', '_source_folder', '_source_file']

🎉 已寫入完成：/content/drive/MyDrive/ZecZec_Group_Data/zeczec_history.db
   資料表名稱：projects_summary（共 2104 筆）


In [4]:

##########查詢db用的

import pandas as pd
import sqlite3

# 定義你的資料庫路徑 (請確認路徑與你剛才存檔的一致)
db_path = "/content/drive/MyDrive/ZecZec_Group_Data/zeczec_history.db"

# 1. 連線到 SQLite 資料庫
conn = sqlite3.connect(db_path)

# 2. 使用 SQL 語法直接撈取前 50 筆資料 (LIMIT 50)
# 這樣做比把整個資料庫讀出來再切 50 筆更節省記憶體
sql_query = "SELECT * FROM projects;"
df_top50 = pd.read_sql_query(sql_query, conn)

# 關閉連線
conn.close()

# 3. 在 Colab 中印出漂亮的表格
print(f"📊 成功撈取前 {len(df_top50)} 筆資料，以下為資料預覽：")
display(df_top50)

📊 成功撈取前 2102 筆資料，以下為資料預覽：


,主分類,次分類,募資狀態,專案編號,方案價格列表,折扣層數,FAQ總題數,FAQ更新頻率,開始日期,結束日期,總天數,達標率(%)
0,群眾募資,出版,成功,PS1,3600 | 18000 | 36000,3,0,0.0000,2026-06-01,2026-06-22,21,1440
1,群眾募資,出版,成功,PS2,2100 | 1400 | 2400 | 3400 | 2100,5,13,0.3171,2026-05-25,2026-07-05,41,126
2,群眾募資,出版,成功,PS3,1280 | 2380 | 1480 | 5680,4,7,0.1556,2026-05-19,2026-07-03,45,1644
3,群眾募資,出版,成功,PS4,1780 | 3360 | 1880 | 1680,4,4,0.0930,2026-05-18,2026-06-30,43,459
4,群眾募資,出版,成功,PS5,1199 | 399 | 3980 | 9999 | 418 | 1900,6,8,0.1290,2026-05-14,2026-07-16,62,3679
...,...,...,...,...,...,...,...,...,...,...,...,...
2097,預購式專案,飲食,失敗,FDF130,2530 | 1400 | 1050,3,3,0.0405,2023-08-25,2023-11-08,74,34
2098,預購式專案,飲食,失敗,FDF131,2195 | 1880,2,6,0.7500,2023-08-22,2023-08-31,8,0
2099,預購式專案,飲食,失敗,FDF132,22680 | 3680,2,2,0.0476,2023-07-18,2023-08-30,42,0
2100,預購式專案,飲食,失敗,FDF133,859 | 1129 | 1200,3,7,0.2059,2023-07-06,2023-08-10,34,31


In [ ]:
#####合併資料用的

import pandas as pd
import os

# 1. 設定檔案路徑 (請確認路徑是否與你的雲端硬碟一致)
base_folder = '/content'
summary_path = os.path.join(base_folder, 'projects_summary.csv')
pmf_path = os.path.join(base_folder, 'Project_PMF_Dataset.csv')
output_path = os.path.join(base_folder, 'Project_PMF_Dataset_with_dates.csv')

print("🔄 開始進行資料合併...")

# 2. 讀取兩個 CSV 檔案
df_summary = pd.read_csv(summary_path)
df_pmf = pd.read_csv(pmf_path)

# 3. 【關鍵步驟】從 PMF 的 Project_Name (例如 ES1_rogerems) 中萃取出前面的專案編號 (ES1)
df_pmf['專案編號'] = df_pmf['Project_Name'].apply(lambda x: str(x).split('_')[0])

# 4. 從 summary 表中，只抽出我們需要的「專案編號」、「開始日期」、「結束日期」這三個欄位
df_summary_dates = df_summary[['專案編號', '開始日期', '結束日期']]

# 5. 將兩個表依照「專案編號」合併 (Left Join：以 PMF 表為主)
df_merged = pd.merge(df_pmf, df_summary_dates, on='專案編號', how='left')

# 6. (可選) 刪除剛剛為了對接而產生出來的「專案編號」暫存欄位，保持原本檔案的乾淨
df_merged = df_merged.drop(columns=['專案編號'])

# 7. 儲存成全新的 CSV 檔案 (使用 utf-8-sig 確保 Excel 打開不會中文亂碼)
df_merged.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✅ 合併大成功！")
print(f"📂 你的新檔案已經熱騰騰的出爐，存在這裡：\n{output_path}")

# 在 Colab 中預覽前 5 筆資料，確認「開始日期」與「結束日期」有沒有成功接上去
display(df_merged.head(5))

🔄 開始進行資料合併...


FileNotFoundError: [Errno 2] No such file or directory: '/content/Project_PMF_Dataset.csv'